# Week 56, Day 6: Debrief

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w056d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


In [ ]:
def longest_with_k_kinds(s, k):
    # pattern: sliding window with a dict of counts; cost: O(n)
    counts, left, best = {}, 0, 0
    for right, ch in enumerate(s):
        counts[ch] = counts.get(ch, 0) + 1
        while len(counts) > k:
            counts[s[left]] -= 1
            if counts[s[left]] == 0:
                del counts[s[left]]
            left += 1
        best = max(best, right - left + 1)
    return best

def count_with_k_kinds(s, k):
    # every window ending at right contributes right - left + 1 substrings
    counts, left, total = {}, 0, 0
    for right, ch in enumerate(s):
        counts[ch] = counts.get(ch, 0) + 1
        while len(counts) > k:
            counts[s[left]] -= 1
            if counts[s[left]] == 0:
                del counts[s[left]]
            left += 1
        total += right - left + 1
    return total

print(longest_with_k_kinds("eceba", 2), count_with_k_kinds("aab", 1))

In [ ]:
from collections import deque

def spoil_times(grid):
    # pattern: BFS from every mouldy fruit at once; cost: O(cells)
    R = len(grid)
    C = len(grid[0]) if R else 0
    times = {(r, c): 0 for r in range(R) for c in range(C) if grid[r][c] == "M"}
    queue = deque(times)
    while queue:
        r, c = queue.popleft()
        for nr, nc in ((r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)):
            if 0 <= nr < R and 0 <= nc < C and grid[nr][nc] == "F" and (nr, nc) not in times:
                times[(nr, nc)] = times[(r, c)] + 1
                queue.append((nr, nc))
    return times

def minutes_until_spoiled(grid):
    times = spoil_times(grid)
    fruit = sum(row.count("F") + row.count("M") for row in grid)
    if len(times) < fruit:
        return -1
    return max(times.values(), default=0)

print(minutes_until_spoiled(["MFF", "FF.", ".FF"]), minutes_until_spoiled(["MFF", ".FF", "F.F"]))

In [ ]:
def fewest_words(text, words):
    # pattern: prefix-split DP with the last word's start; cost: O(n × longest word)
    longest = max(map(len, words), default=0)
    n = len(text)
    INF = float("inf")
    best, start = [0] + [INF] * n, [None] * (n + 1)
    for i in range(1, n + 1):
        for j in range(max(0, i - longest), i):
            if best[j] + 1 < best[i] and text[j:i] in words:
                best[i], start[i] = best[j] + 1, j
    if best[n] == INF:
        return None
    out, i = [], n
    while i:
        out.append(text[start[i]:i])
        i = start[i]
    return out[::-1]

print(fewest_words("pineapplepen", {"pine", "apple", "pineapple", "pen", "applepen"}))

In [ ]:
import heapq

class KthBest:
    # pattern: a min-heap of the k best; cost: O(log k) per add
    def __init__(self, k, scores=()):
        self.k, self.heap = k, []
        for s in scores:
            self.add(s)

    def add(self, score):
        if len(self.heap) < self.k:
            heapq.heappush(self.heap, score)
        elif score > self.heap[0]:
            heapq.heapreplace(self.heap, score)
        return self.heap[0] if len(self.heap) == self.k else None

    def top(self):
        return sorted(self.heap, reverse=True)

board = KthBest(3, [4, 5, 8, 2])
print([board.add(x) for x in (3, 5, 10, 9, 4)], board.top())

In [ ]:
def largest_in_histogram(heights):
    # pattern: monotonic stack of increasing heights; cost: O(n)
    best, stack = 0, []                     # indices with increasing heights
    for i, h in enumerate(list(heights) + [0]):
        while stack and heights[stack[-1]] >= h:
            height = heights[stack.pop()]
            left = stack[-1] + 1 if stack else 0
            best = max(best, height * (i - left))
        stack.append(i)
    return best

def largest_block(grid):
    if not grid:
        return 0
    cols = [0] * len(grid[0])
    best = 0
    for row in grid:
        cols = [c + 1 if ch == "1" else 0 for c, ch in zip(cols, row)]
        best = max(best, largest_in_histogram(cols))
    return best

print(largest_in_histogram([2, 1, 5, 6, 2, 3]), largest_block(["10100", "10111", "11111", "10010"]))

In [ ]:
import bisect

def longest_rising(nums):
    tops, tops_idx, parent = [], [], [None] * len(nums)
    for i, x in enumerate(nums):
        k = bisect.bisect_left(tops, x)
        parent[i] = tops_idx[k - 1] if k > 0 else None
        if k == len(tops):
            tops.append(x)
            tops_idx.append(i)
        else:
            tops[k], tops_idx[k] = x, i
    out, i = [], tops_idx[-1] if tops_idx else None
    while i is not None:
        out.append(nums[i])
        i = parent[i]
    return out[::-1]

print(longest_rising([10, 9, 2, 5, 3, 7, 101, 18, 7]))

In [ ]:
def _lakes(grid):
    R = len(grid)
    C = len(grid[0]) if R else 0
    seen, sizes = set(), []
    for r in range(R):
        for c in range(C):
            if grid[r][c] != "W" or (r, c) in seen:
                continue
            size, inland, todo = 0, True, [(r, c)]
            seen.add((r, c))
            while todo:
                a, b = todo.pop()
                size += 1
                if a in (0, R - 1) or b in (0, C - 1):
                    inland = False
                for x, y in ((a + 1, b), (a - 1, b), (a, b + 1), (a, b - 1)):
                    if 0 <= x < R and 0 <= y < C and grid[x][y] == "W" and (x, y) not in seen:
                        seen.add((x, y))
                        todo.append((x, y))
            if inland:
                sizes.append(size)
    return sizes

def count_lakes(grid):
    # pattern: flood fill with a stack; cost: O(cells)
    return len(_lakes(grid))

def largest_lake(grid):
    return max(_lakes(grid), default=0)

print(count_lakes(["LLLLL", "LWWLL", "LWLWL", "LLLLW", "WLWLL"]), largest_lake(["LLLLL", "LWWLL", "LWLWL", "LLLLW", "WLWLL"]))

In [ ]:
def _rounds(n, flights, src, k):
    INF = float("inf")
    cost = [INF] * n
    cost[src] = 0
    history = [(list(cost), [None] * n)]
    for _ in range(k + 1):
        prev = history[-1][0]
        new, came = list(prev), [None] * n
        for a, b, p in flights:
            if prev[a] + p < new[b]:
                new[b], came[b] = prev[a] + p, a
        history.append((new, came))
    return history

def cheapest_with_stops(n, flights, src, dst, k):
    # pattern: Bellman-Ford limited to k + 1 rounds, each from the previous round's costs
    final = _rounds(n, flights, src, k)[-1][0][dst]
    return -1 if final == float("inf") else final

def route_with_stops(n, flights, src, dst, k):
    history = _rounds(n, flights, src, k)
    if history[-1][0][dst] == float("inf"):
        return None
    route, city, r = [dst], dst, len(history) - 1
    while r > 0:
        cost, came = history[r]
        if history[r - 1][0][city] == cost[city]:
            r -= 1                                # the same cost was already reachable a round earlier
            continue
        city = came[city]
        route.append(city)
        r -= 1
    return route[::-1]

f = [(0, 1, 100), (1, 2, 100), (2, 3, 100), (0, 2, 500), (1, 3, 600)]
print(cheapest_with_stops(4, f, 0, 3, 1), route_with_stops(4, f, 0, 3, 2))

In [ ]:
def combinations_summing(candidates, target):
    # pattern: backtracking, sorted, skipping equal values at one level; prune on size
    nums = sorted(candidates)
    out, path = [], []

    def go(start, left):
        if left == 0:
            out.append(list(path))
            return
        for i in range(start, len(nums)):
            if nums[i] > left:
                break
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            go(i + 1, left - nums[i])
            path.pop()

    go(0, target)
    return sorted(out)

print(combinations_summing([10, 1, 2, 7, 6, 1, 5], 8))

In [ ]:
def riffle_choices(a, b, c):
    # pattern: DP over prefix lengths (i of a, j of b); cost: O(len(a) × len(b))
    n, m = len(a), len(b)
    if n + m != len(c):
        return 0
    ways = [[0] * (m + 1) for _ in range(n + 1)]
    ways[0][0] = 1
    for i in range(n + 1):
        for j in range(m + 1):
            if i and a[i - 1] == c[i + j - 1]:
                ways[i][j] += ways[i - 1][j]
            if j and b[j - 1] == c[i + j - 1]:
                ways[i][j] += ways[i][j - 1]
    return ways[n][m]

def is_interleaving(a, b, c):
    return riffle_choices(a, b, c) > 0

print(is_interleaving("aabcc", "dbbca", "aadbbcbcac"), riffle_choices("a", "a", "aa"))

In [ ]:
def smallest_team(skills, people):
    # pattern: DP over skill masks; cost: O(2^skills × people)
    index = {s: i for i, s in enumerate(skills)}
    full = (1 << len(skills)) - 1
    masks = []
    for p in people:
        m = 0
        for s in p:
            if s in index:
                m |= 1 << index[s]
        masks.append(m)
    best = {0: []}
    for mask in range(full + 1):
        if mask not in best:
            continue
        team = best[mask]
        for i, m in enumerate(masks):
            new = mask | m
            if new != mask and (new not in best or len(best[new]) > len(team) + 1):
                best[new] = team + [i]
    return sorted(best[full]) if full in best else None

print(smallest_team(["java", "nodejs", "reactjs"], [["java"], ["nodejs"], ["nodejs", "reactjs"]]))

In [ ]:
def cheapest_network(n, cables):
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            x = parent[x]
        return x
    total, joined = 0, 0
    for a, b, cost in sorted(cables, key=lambda c: c[2]):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[ra] = rb
            total += cost
            joined += 1
    return total if joined == n - 1 else None

cables = [(0, 1, 4), (1, 2, 1), (0, 2, 3), (2, 3, 2), (1, 3, 5)]
print([cheapest_network(4, cables), cheapest_network(5, cables)])

### Exercise: Second chance: a budget window

Positive whole numbers are the prices of items on a shelf. Write `longest_affordable(prices, budget)`: the most neighbouring items you can buy with at most `budget`, in O(n), and `count_affordable(prices, budget)`: how many non-empty runs of neighbouring items cost at most the budget.

In [ ]:
def longest_affordable(prices, budget):
    ___

def count_affordable(prices, budget):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_window():\n    \"a window again\"\n    assert longest_affordable([2, 1, 3, 4, 1], 5) == 2 and longest_affordable([6], 5) == 0 and longest_affordable([], 3) == 0\n    assert count_affordable([1, 2, 3], 3) == 4\n    rng = random.Random(1)\n    for _ in range(200):\n        p = [rng.randint(1, 6) for _ in range(rng.randint(0, 10))]\n        b = rng.randint(0, 15)\n        runs = [j - i for i in range(len(p)) for j in range(i + 1, len(p) + 1) if sum(p[i:j]) <= b]\n        assert longest_affordable(p, b) == max(runs, default=0) and count_affordable(p, b) == len(runs)\n    assert count_affordable([1] * 100_000, 10) > 0")

### Exercise: Second chance: the nearest exit

A floor plan: `"#"` wall, `"E"` exit, `"."` floor. Write `exit_distances(grid)`: a list of lists with, for each floor or exit cell, the fewest steps (up, down, left, right) to the nearest exit, and `None` for walls and for floor that can't reach one. Multi-source BFS.

In [ ]:
from collections import deque

def exit_distances(grid):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_gates():\n    \"multi-source BFS\"\n    assert exit_distances([\"E.#\", \"..#\", \"#.E\"]) == [[0, 1, None], [1, 2, None], [None, 1, 0]]\n    assert exit_distances([\"..\", \"..\"]) == [[None, None], [None, None]] and exit_distances([\"E\"]) == [[0]]\n    rng = random.Random(2)\n    for _ in range(80):\n        R, C = rng.randint(1, 5), rng.randint(1, 5)\n        grid = [\"\".join(rng.choice(\"..#E\") for _ in range(C)) for _ in range(R)]\n        got = exit_distances(grid)\n        for r in range(R):\n            for c in range(C):\n                if grid[r][c] == \"#\":\n                    assert got[r][c] is None\n                    continue\n                dist, todo, seen, found = 0, [(r, c)], {(r, c)}, None\n                while todo and found is None:\n                    if any(grid[a][b] == \"E\" for a, b in todo):\n                        found = dist\n                        break\n                    nxt = []\n                    for a, b in todo:\n                        for x, y in ((a + 1, b), (a - 1, b), (a, b + 1), (a, b - 1)):\n                            if 0 <= x < R and 0 <= y < C and grid[x][y] != \"#\" and (x, y) not in seen:\n                                seen.add((x, y))\n                                nxt.append((x, y))\n                    todo, dist = nxt, dist + 1\n                assert got[r][c] == found")

### Exercise: Second chance: how many splits?

Write `split_ways(text, words)`: the number of ways to split the text into dictionary words (words may repeat), mod 10⁹ + 7, with a prefix-split DP.

In [ ]:
def split_ways(text, words):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom functools import cache\ndef test_split():\n    \"counting splits\"\n    assert split_ways(\"catsanddog\", {\"cat\", \"cats\", \"and\", \"sand\", \"dog\"}) == 2 and split_ways(\"\", {\"a\"}) == 1\n    assert split_ways(\"abc\", {\"ab\"}) == 0 and split_ways(\"aaaa\", {\"a\", \"aa\"}) == 5\n    rng = random.Random(3)\n    for _ in range(100):\n        words = frozenset(\"\".join(rng.choice(\"ab\") for _ in range(rng.randint(1, 3))) for _ in range(4))\n        text = \"\".join(rng.choice(\"ab\") for _ in range(rng.randint(0, 12)))\n        @cache\n        def go(i):\n            return 1 if i == len(text) else sum(go(j) for j in range(i + 1, len(text) + 1) if text[i:j] in words)\n        assert split_ways(text, words) == go(0)\n    assert split_ways(\"a\" * 1000, {\"a\", \"aa\"}) > 0")

### Exercise: Second chance: trapped rain

Bars of the given heights stand side by side. Write `trapped_water(heights)`: how much rain they hold, with a monotonic stack (pop a bar when a taller one arrives: the water above it is bounded by the bar below it on the stack and the new bar).

In [ ]:
def trapped_water(heights):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_rain():\n    \"a stack again\"\n    assert trapped_water([0, 1, 0, 2, 1, 0, 1, 3, 2, 1, 2, 1]) == 6 and trapped_water([4, 2, 0, 3, 2, 5]) == 9\n    assert trapped_water([]) == 0 and trapped_water([3]) == 0 and trapped_water([1, 2, 3]) == 0\n    rng = random.Random(4)\n    for _ in range(200):\n        h = [rng.randint(0, 6) for _ in range(rng.randint(0, 12))]\n        want = sum(max(0, min(max(h[:i + 1]), max(h[i:])) - h[i]) for i in range(len(h)))\n        assert trapped_water(h) == want")

### Exercise: Second chance: covering every shift

A rota has up to 12 shifts that must all be covered. Each volunteer can cover some shifts and asks a fee. Write `cheapest_cover(shifts, volunteers)`: the least total fee for a group covering every shift, or `None`. `volunteers` is a list of `(fee, list_of_shifts)`. DP over masks of shifts covered.

In [ ]:
def cheapest_cover(shifts, volunteers):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import itertools, random\ndef test_cover():\n    \"a mask of shifts\"\n    v = [(5, [\"mon\", \"tue\"]), (3, [\"tue\"]), (4, [\"wed\"]), (6, [\"mon\", \"wed\"])]\n    assert cheapest_cover([\"mon\", \"tue\", \"wed\"], v) == 9 and cheapest_cover([\"thu\"], v) is None and cheapest_cover([], v) == 0\n    rng = random.Random(5)\n    names = [f\"s{i}\" for i in range(5)]\n    for _ in range(60):\n        need = rng.sample(names, rng.randint(1, 5))\n        vols = [(rng.randint(1, 9), rng.sample(names, rng.randint(1, 3))) for _ in range(rng.randint(1, 6))]\n        best = None\n        for k in range(len(vols) + 1):\n            for team in itertools.combinations(vols, k):\n                if set(need) <= {s for _, ss in team for s in ss}:\n                    cost = sum(f for f, _ in team)\n                    best = cost if best is None else min(best, cost)\n        assert cheapest_cover(need, vols) == best")

### Bug hunt: Second chance: three more bugs

`fewest_jumps(stones, k)` should return the fewest jumps from stone 0 to the last stone, where you may jump from stone i to any stone j with i < j ≤ i + k **that is safe** (`stones[j]` is True); `None` if impossible. It's a BFS with three bugs: it uses a stack, it marks stones as seen only when they're taken off (so they're added many times), and it lets you land on unsafe stones. Fix them so this prints `[3, None, 4]`.

In [ ]:
def fewest_jumps(stones, k):
    n = len(stones)
    todo, seen, dist = [0], set(), {0: 0}
    while todo:
        i = todo.pop()
        if i in seen:
            continue
        seen.add(i)
        for j in range(i + 1, min(n, i + k + 1)):
            if j not in dist or dist[j] > dist[i] + 1:
                dist[j] = dist[i] + 1
                todo.append(j)
    return dist.get(n - 1)

S, U = True, False
print([fewest_jumps([S, S, U, S, S], 2), fewest_jumps([S, U, U, S], 2), fewest_jumps([S, S, U, S, U, S, S], 2)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"fixed\"\n    expect_output(\"[3, None, 4]\")\n    assert fewest_jumps([True], 3) == 0 and fewest_jumps([True, True, True, True], 1) == 3")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Your scores

Write down your totals for timed sets 3, 4 and 5 before reading on. For every lost mark, note **why**:

1. **Didn't know it**: a gap in what you've learned. Revise the topic.
2. **Picked the wrong pattern**: you had the tools but reached for the wrong one. Note what in the problem should have pointed you to the right one.
3. **Edge case**: the main case worked, but empty input, a tie or a boundary failed.
4. **Ran out of time**: the plan, not the knowledge, was the problem.

| Score (sets 4 and 5) | What it means |
|---|---|
| 80 or more | Excellent: you can use Phase 4 under pressure. |
| 60 to 79 | Good. Practise the patterns where you lost marks. |
| under 60 | Not yet. Spend extra time on the second chances and next week's review days, then retry the sets without looking at the solutions. |

## Part A, both sets

| Set 4 | The idea | Revise |
|---|---|---|
| 1 | prefix sums: `prefix[j] - prefix[i]` is `nums[i:j]` | [Week 41, Day 5](w041d5.html) |
| 2 | monotonic deques for window maximums | [Week 43, Day 3](w043d3.html) |
| 3 | balanced trees have height about log₂ n | [Week 46, Day 1](w046d1.html) |
| 4 | a min-heap pops smallest first | [Week 47, Day 1](w047d1.html) |
| 5 | Bellman-Ford handles negative edges | [Week 51, Day 2](w051d2.html) |
| 6 | union-find detects cycles in Kruskal | [Week 51, Day 4](w051d4.html) |
| 7 | memoised Fibonacci-style counting | [Week 53, Day 1](w053d1.html) |
| 8 | 0/1 knapsack loops capacity downwards | [Week 53, Day 5](w053d5.html) |
| 9 | x & −x isolates the lowest set bit | [Week 55, Day 2](w055d2.html) |
| 10 | C(n, k) counts subsets | [Week 55, Day 4](w055d4.html) |

| Set 5 | The idea | Revise |
|---|---|---|
| 1 | BFS states must include everything that matters | [Week 51, Day 5](w051d5.html) |
| 2 | Bellman-Ford with k + 1 rounds limits the stops | [Week 51, Day 2](w051d2.html) |
| 3 | backtracking lists every subset, depth first | [Week 52, Day 3](w052d3.html) |
| 4 | edit-distance edges | [Week 54, Day 1](w054d1.html) |
| 5 | the LCS table on prefixes | [Week 54, Day 1](w054d1.html) |
| 6 | Held-Karp's 2ⁿ × n states | [Week 54, Day 4](w054d4.html) |
| 7 | the submask loop | [Week 54, Day 4](w054d4.html) |
| 8 | the product rule and stars and bars | [Week 55, Day 4](w055d4.html) |
| 9 | Monte Carlo versus Las Vegas | [Week 55, Day 5](w055d5.html) |
| 10 | powers of two and the lowest set bit | [Week 55, Day 2](w055d2.html) |

## Part B, both sets

Each model solution runs, so try it. The notes give the key decision and the most common mistakes.

### B1: At most k kinds

**The decision**: a sliding window with a dict of counts; shrink from the left while there are more than k kinds. Counting substrings uses the same window: every window ending at `right` contributes `right - left + 1` substrings. **Common mistakes**: recomputing `len(set(s[left:right + 1]))` each step (O(n²)); forgetting to delete a count that reaches 0, so `len(counts)` never falls.

### B2: Spreading mould

**The decision**: breadth-first search from **every** mouldy fruit at once (multi-source BFS): the minute a fruit spoils is its distance from the nearest mouldy one. **Common mistakes**: running one BFS per mouldy fruit (slower and harder to combine); returning 0 instead of -1 when fresh fruit is cut off; counting empty cells.

### B3: The fewest words

**The decision**: a prefix-split DP (Week 53 Day 3), best[i] = fewest words for text[:i], storing where the last word starts for the walk back; only look back as far as the longest word. **Common mistakes**: greedily taking the longest word first (fails when it blocks the rest); backtracking without memoisation (exponential on "ab" × 1000).

### B4: The kth best score so far

**The decision**: a min-heap of the k best: its top is the k-th best. A new score only enters if it beats the top. **Common mistakes**: a max-heap of everything (O(log n) and growing memory); sorting after every add (O(n log n) each).

### B5: The biggest rectangle

**The decision**: a monotonic stack of increasing heights (Week 43). When a lower bar arrives, every taller bar on the stack has found its right edge; its left edge is the bar below it on the stack. A sentinel 0 at the end flushes the stack. For the grid, each row turns into a histogram of column heights. **Common mistakes**: getting the left edge wrong (it's the index after the new stack top, not the popped index); forgetting the sentinel.

### B6: Three bugs

**The bugs**: `bisect_right` lets equal values extend a run (strict needs `bisect_left`); without `parent[i] = tops_idx[k - 1]` the walk back stops after one element; the walk goes from the end, so the list must be reversed.

### C1: Lakes that don't reach the sea

**The decision**: flood fill each water group with a stack, noting whether any cell touches the edge. **Common mistakes**: recursion, which overflows on the 400 × 400 map; marking cells as seen only when popped, so they're pushed many times.

### C2: Cheap flights, few changes

**The decision**: Bellman-Ford limited to k + 1 rounds (Week 51 Day 2), each round relaxing from the **previous** round's costs only, so no route uses more flights than the round number. Keeping each round's table allows the walk back. **Common mistakes**: Dijkstra, which finds the cheapest route regardless of stops; relaxing in place within a round, which lets one round use several flights.

### C3: Combinations that add up

**The decision**: backtracking over sorted candidates (Week 52 Day 3), skipping a value equal to the previous one **at the same level**, and breaking out once a value exceeds what's left. **Common mistakes**: generating everything and deduplicating with a set (slow); skipping duplicates at every level, which loses [1, 1, 6].

### C4: Shuffled together

**The decision**: a table over prefixes of both decks (Week 54 Day 1's shape): the last card of c[:i + j] came from a or from b. **Common mistakes**: a greedy two-pointer walk, which fails when both decks offer the same card; forgetting the length check.

### C5: The smallest team

**The decision**: DP over masks of skills covered (Week 54 Day 4), best[mask] = the smallest team reaching exactly those skills, extended by each person. With 16 skills that's 65,536 masks × 60 people. **Common mistakes**: trying every team of people (2⁶⁰); greedily taking the person with the most new skills (not always smallest).

### C6: Three bugs

**The bugs**: Kruskal sorts by cost (`key=lambda c: c[2]`); it must join the **roots**, `parent[find(a)] = find(b)`, or trees get cut apart; and it must count the cables used: fewer than n − 1 means not everything is connected.

## Second chances

A fresh problem for each skill the sets tested. If you lost marks on a problem above, do its partner here without looking back.

## Quiz

## What's next

Next week is the **DSA exam week**: two review days (structures, then algorithms), a practice exam, a day of drills on your weak spots, the exam itself and a debrief. Before Monday:

1. List the three patterns that cost you the most marks this week.
2. For each, find the day where you met it (the pattern map from Monday helps) and redo two of its exercises.
3. Retry any timed-set problem you didn't finish, without looking at its model solution first.

Then a rest week, and Phase 5: **Advanced Python**, starting with iterators and generators.